# Purpose:
- Calculate and save codings cores
    - Using session model
    - From 250124_adjusted_variance_explained.ipynb

In [1]:
import sys
sys.path.append('/root/capsule/code/')

from DesignMatrix import DesignMatrix
import glm_fit_tools as gft
import design_matrix_tools as dmtools
import kernel_tools as ktools
import load_data

import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import json

from matplotlib import pyplot as plt

# notebook dev
%load_ext autoreload
%autoreload 2
%matplotlib inline
import capsule_utils



In [2]:
def generate_session_model_adjusted_variance_explained(glm_results, run_params, X_trim, response_trim_filtered):
    
    # building session model
    mean_W = glm_results['W_cv'].mean(dim='test_fold_ind')
    ve_session_model = glm_results['var_explained_mean_model']
    
    # calculating adjusted variance explained for each model
    adj_var_explained_session_model = xr.full_like(ve_session_model,
                                               fill_value=np.nan).drop_sel(model=['Full', 'intercept'])
    adj_var_explained_session_model_full_mask = xr.full_like(ve_session_model,
                                                        fill_value=np.nan).drop_sel(model=['Full', 'intercept'])

    all_weights = mean_W.weights.values
    all_models = adj_var_explained_session_model.model.values
    for model in all_models:
        run_param = run_params['dropouts'][model]
        if run_param['is_single']:
            kernels = run_param['kernels']
        else:
            kernels = run_param['dropped_kernels']
        # remove intercept from kernels for the mask
        kernels = np.setdiff1d(kernels, ['intercept'])
        
        mask_weights = [w for w in all_weights if np.any([mk in w for mk in kernels])]
        mask_trim = (X_trim.sel(weights=mask_weights)!=0).any(dim='weights')
        if mask_trim.sum() == X_trim.sizes['timestamps']:
            adj_var_explained_session_model.loc[{'model': model}] = ve_session_model.sel(model=model)
            adj_var_explained_session_model_full_mask.loc[{'model': model}] = ve_session_model.sel(model='Full')
        else:
            if run_param['is_single']:
                run_weights = ['intercept_0', *mask_weights]
            else:
                run_weights = np.setdiff1d(all_weights, mask_weights)
            
            mask_cv = (X_trim.sel(weights=mask_weights)!=0).any(dim='weights')
            W_model = mean_W.sel(model=model, weights=run_weights)
            X_model = X_trim.sel(weights=run_weights)
            assert np.isnan(W_model).any() == False
            
            adj_var_explained_session_model.loc[{'model': model}] = \
                gft.compute_adjusted_variance_explained(response_trim_filtered, W_model, X_model, mask_cv)
            
            W_full = mean_W.sel(model='Full')
        
            adj_var_explained_session_model_full_mask.loc[{'model': model}] = \
                gft.compute_adjusted_variance_explained(response_trim_filtered, W_full, X_trim, mask_cv)
    return adj_var_explained_session_model, adj_var_explained_session_model_full_mask


def calculate_coding_score(adj_var_explained_session_model, adj_var_explained_session_model_full_mask, run_params,
                           filter_threshold=0.005):
    coding_score_session_model = xr.full_like(adj_var_explained_session_model, fill_value=np.nan)
    all_models = adj_var_explained_session_model.model.values
    single_models = [m for m in all_models if run_params['dropouts'][m]['is_single']]
    dropout_models = [m for m in all_models if not run_params['dropouts'][m]['is_single']]

    coding_score_session_model.loc[{'model': single_models}] = -adj_var_explained_session_model.sel(model=single_models) / adj_var_explained_session_model_full_mask.sel(model=single_models)
    coding_score_session_model.loc[{'model': dropout_models}] = -(1 - (adj_var_explained_session_model.sel(model=dropout_models) / adj_var_explained_session_model_full_mask.sel(model=dropout_models)))

    # filtering
    coding_score_session_model_filtered_before_clipping = coding_score_session_model.copy()
    session_adjVE_mask = adj_var_explained_session_model < filter_threshold
    session_adjVE_full_mask_mask = adj_var_explained_session_model_full_mask < filter_threshold
    coding_score_session_model_filtered_before_clipping = coding_score_session_model_filtered_before_clipping.where(
        ~(session_adjVE_mask + session_adjVE_full_mask_mask), other=0)

    # clipping
    coding_score_session_model_filtered = coding_score_session_model_filtered_before_clipping.clip(min=-1, max=0)
    
    return coding_score_session_model_filtered

In [3]:
data_type = 'events'
dm_version = 2
suffix = ''
load_path = Path('/root/capsule/scratch/Thyme')
save_dir = load_path

data_dir = '/root/capsule/data'
data_folders = [d for d in glob.glob(data_dir + '/*') if Path(d).is_dir()]
raw_paths = np.sort([d for d in data_folders if ('processed' not in d.split('/')[-1]) and
                    ('dlc-eye' not in d.split('/')[-1]) and
                    ('multiplane-ophys' in d.split('/')[-1]) and 
                    ('stimuli' not in d.split('/')[-1]) and
                    ('stim-response' not in d.split('/')[-1]) and
                    ('ROICat' not in d.split('/')[-1])])
session_names = [d.split('/')[-1] for d in raw_paths]
session_names

['multiplane-ophys_736963_2024-07-24_08-49-57',
 'multiplane-ophys_736963_2024-07-26_09-37-45',
 'multiplane-ophys_736963_2024-07-29_09-00-58',
 'multiplane-ophys_736963_2024-07-30_09-11-03',
 'multiplane-ophys_736963_2024-08-01_08-59-00',
 'multiplane-ophys_736963_2024-08-05_09-19-25',
 'multiplane-ophys_736963_2024-08-06_08-52-03',
 'multiplane-ophys_736963_2024-08-07_09-11-10',
 'multiplane-ophys_736963_2024-08-09_08-58-36',
 'multiplane-ophys_736963_2024-08-12_09-17-19',
 'multiplane-ophys_736963_2024-08-13_08-57-29']

# One session example

In [15]:
session_name = session_names[5]
glm_fn = load_path / f'glm_results_v{dm_version:02}_{session_name}_{data_type}{suffix}.npy'
glm_results = np.load(glm_fn, allow_pickle=True).item()

X, response, response_info, run_params, unstd_features, use_indices = \
    gft.load_data(session_name, data_type, dm_version, load_path=load_path)

# trim X and response based on the shift
X_trim = X[use_indices, :]
response_trim = response[use_indices, :]
response_trim_filtered = gft.filter_response_matrix(response_trim)

adj_var_explained_session_model, adj_var_explained_session_model_full_mask = \
    generate_session_model_adjusted_variance_explained(glm_results, run_params, X_trim, response_trim_filtered)
coding_score = calculate_coding_score(adj_var_explained_session_model, adj_var_explained_session_model_full_mask, run_params)

In [16]:
adj_var_explained_session_model

<xarray.DataArray (model: 34, cell_roi_id: 617)> Size: 168kB
array([[ 0.04846185,  0.086018  ,  0.24999747, ...,  0.02221863,
        -0.01519614,  0.0654656 ],
       [ 0.05513881,  0.1334777 ,  0.11002692, ...,  0.05644299,
         0.22742667,  0.17373943],
       [ 0.03724341,  0.08710636,  0.14463382, ...,  0.02065301,
         0.09227028,  0.0324475 ],
       ...,
       [ 0.01525257,  0.04528026,  0.15808886, ...,  0.024033  ,
         0.01585987,  0.00397284],
       [ 0.06306783,  0.0969607 ,  0.24961349, ...,  0.02419192,
         0.0391995 ,  0.05360734],
       [ 0.0254845 ,  0.05217609,  0.03197067, ...,  0.01238856,
         0.06096628,  0.03891909]])
Coordinates:
  * model        (model) object 272B 'hits' 'misses' ... 'single-behavioral'
  * cell_roi_id  (cell_roi_id) object 5kB 'VISp_7_0000' ... 'VISp_6_0057'

In [29]:
# make a dataset from dataarrays

adj_var_explained_session_model = adj_var_explained_session_model.expand_dims("metric").assign_coords(metric=["adj_ve_model"])
adj_var_explained_session_model_full_mask = adj_var_explained_session_model_full_mask.expand_dims("metric").assign_coords(metric=["adj_ve_full"])
coding_score = coding_score.expand_dims("metric").assign_coords(metric=["coding_score"])

cs_metrics = xr.concat([adj_var_explained_session_model, adj_var_explained_session_model_full_mask, coding_score], dim="metric")
cs_metrics.name = 'coding_score_metrics_from_session_model'
cs_metrics


<xarray.DataArray 'coding_score_metrics_from_session_model' (metric: 3,
                                                             model: 34,
                                                             cell_roi_id: 617)> Size: 503kB
array([[[ 0.04846185,  0.086018  ,  0.24999747, ...,  0.02221863,
         -0.01519614,  0.0654656 ],
        [ 0.05513881,  0.1334777 ,  0.11002692, ...,  0.05644299,
          0.22742667,  0.17373943],
        [ 0.03724341,  0.08710636,  0.14463382, ...,  0.02065301,
          0.09227028,  0.0324475 ],
        ...,
        [ 0.01525257,  0.04528026,  0.15808886, ...,  0.024033  ,
          0.01585987,  0.00397284],
        [ 0.06306783,  0.0969607 ,  0.24961349, ...,  0.02419192,
          0.0391995 ,  0.05360734],
        [ 0.0254845 ,  0.05217609,  0.03197067, ...,  0.01238856,
          0.06096628,  0.03891909]],

       [[ 0.08268555,  0.15738639,  0.32785488, ...,  0.03812215,
          0.02661541,  0.07348688],
        [ 0.07493517,  0.1751372 ,  0.19188024, ...,  0.06843342,
          0.25156655,  0.17976546],
        [ 0.05059799,  0.09330203,  0.17303796, ...,  0.02901809,
          0.09346909,  0.04578172],
...
        [ 0.04746947,  0.10899579,  0.19185163, ...,  0.03896522,
          0.08369422,  0.04608624],
        [ 0.08213446,  0.15894458,  0.32064214, ...,  0.04497425,
          0.05590543,  0.07518722],
        [ 0.04671002,  0.10719838,  0.19107539, ...,  0.03801327,
          0.08475788,  0.04651746]],

       [[-0.41390178, -0.45345975, -0.23747522, ..., -0.41717269,
          0.        , -0.10915249],
        [-0.26417979, -0.23786782, -0.42658549, ..., -0.17521304,
         -0.09595821, -0.03352161],
        [-0.263935  , -0.06640446, -0.16414979, ..., -0.28827118,
         -0.0128258 , -0.29125637],
        ...,
        [-0.32131315, -0.41543127, -0.82401629, ..., -0.61678085,
         -0.18949774,  0.        ],
        [-0.76786078, -0.61002836, -0.77847998, ..., -0.53790601,
         -0.70117516, -0.71298476],
        [-0.54558947, -0.48672458, -0.16731968, ..., -0.32590093,
         -0.71929921, -0.83665542]]])
Coordinates:
  * model        (model) object 272B 'hits' 'misses' ... 'single-behavioral'
  * cell_roi_id  (cell_roi_id) object 5kB 'VISp_7_0000' ... 'VISp_6_0057'
  * metric       (metric) <U12 144B 'adj_ve_model' 'adj_ve_full' 'coding_score'

In [30]:
# saving datasets aggregated from adj_var_explained_session_model, adj_var_explained_session_model_full_mask, coding_score
save_dir = load_path
save_fn = save_dir / f'coding_score_v{dm_version:02}_{session_name}_{data_type}{suffix}.nc'
cs_metrics.to_netcdf(save_fn)



In [31]:
with xr.open_dataset(save_fn) as nc:
    cs_metrics = nc

# Run on multiple sessions

In [4]:
#####
## check dm_version, data_type, and suffix
#####

save_dir = load_path / f'coding_score_736963_natural_images_v{dm_version:02}_{data_type}'
save_dir.mkdir(exist_ok=True, parents=True)
for session_name in session_names:
    save_fn = save_dir / f'coding_score_v{dm_version:02}_{session_name}_{data_type}{suffix}.nc'
    if os.path.exists(save_fn):
        print(f"Skipping {session_name}")
        continue
    print(f"Processing {session_name}")
    
    glm_fn = load_path / f'glm_results_v{dm_version:02}_{session_name}_{data_type}{suffix}.npy'
    glm_results = np.load(glm_fn, allow_pickle=True).item()

    X, response, response_info, run_params, unstd_features, use_indices = \
        gft.load_data(session_name, data_type, dm_version, load_path=load_path)

    # trim X and response based on the shift
    X_trim = X[use_indices, :]
    response_trim = response[use_indices, :]
    response_trim_filtered = gft.filter_response_matrix(response_trim)

    # Calculate adjusted variance explained and coding score
    adj_var_explained_session_model, adj_var_explained_session_model_full_mask = \
        generate_session_model_adjusted_variance_explained(glm_results, run_params, X_trim, response_trim_filtered)
    coding_score = calculate_coding_score(adj_var_explained_session_model, adj_var_explained_session_model_full_mask, run_params)
    
    # Merge them into a single dataset
    adj_var_explained_session_model = adj_var_explained_session_model.expand_dims("metric").assign_coords(metric=["adj_ve_model"])
    adj_var_explained_session_model_full_mask = adj_var_explained_session_model_full_mask.expand_dims("metric").assign_coords(metric=["adj_ve_full"])
    coding_score = coding_score.expand_dims("metric").assign_coords(metric=["coding_score"])
    cs_metrics = xr.concat([adj_var_explained_session_model, adj_var_explained_session_model_full_mask, coding_score], dim="metric")
    cs_metrics.name = 'coding_score_metrics_from_session_model'
    
    # saving
    cs_metrics.to_netcdf(save_fn)

Processing multiplane-ophys_736963_2024-07-24_08-49-57
Processing multiplane-ophys_736963_2024-07-26_09-37-45
Processing multiplane-ophys_736963_2024-07-29_09-00-58
Processing multiplane-ophys_736963_2024-07-30_09-11-03
Processing multiplane-ophys_736963_2024-08-01_08-59-00
Processing multiplane-ophys_736963_2024-08-05_09-19-25
Processing multiplane-ophys_736963_2024-08-06_08-52-03
Processing multiplane-ophys_736963_2024-08-07_09-11-10
Processing multiplane-ophys_736963_2024-08-09_08-58-36
Processing multiplane-ophys_736963_2024-08-12_09-17-19
Processing multiplane-ophys_736963_2024-08-13_08-57-29


# Check NaN values
- for adj_ve_model and adj_ve_full_mask

In [39]:
cs_fn = '/root/capsule/scratch/Thyme/coding_score_736963_natural_images_v01_events/coding_score_v01_multiplane-ophys_736963_2024-08-05_09-19-25_events_00.nc'
with xr.open_dataarray(cs_fn) as cs:
    cs_metrics = cs

In [40]:
np.where(cs_metrics.isnull())

(array([], dtype=int64), array([], dtype=int64), array([], dtype=int64))